# P3: Norm Classifier — Data Exploration

This notebook performs EDA on the merged dataset built from:
- **CultureBank** (Reddit + TikTok) → norm sentences
- **GenericsKB** (SimpleWiki + Best) → non-norm sentences

**Sections:**
1. Setup & Load Data
2. Dataset Overview
3. Label & Source Distribution
4. Sentence Length Analysis
5. Cultural Group Analysis
6. Word Frequency Analysis
7. Word Clouds
8. Train / Val / Test Split Overview
9. Sample Sentences

## 1. Setup & Load Data

In [ ]:
import os, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

# optional — install with: pip install wordcloud
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print("wordcloud not installed. Run: pip install wordcloud")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

NORM_COLOR    = "#2196F3"
NONNORM_COLOR = "#FF7043"

# ── paths (notebook is inside PROJECT_P3/notebooks/) ──
DATA_DIR = os.path.join("..", "data")

merged = pd.read_csv(os.path.join(DATA_DIR, "merged_full.csv"))
train  = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val    = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test   = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Merged : {len(merged):,} rows")
print(f"Train  : {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Columns: {merged.columns.tolist()}")

## 2. Dataset Overview

In [ ]:
print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(f"\n  Total samples  : {len(merged):,}")
print(f"  Norm     (1)   : {(merged['label']==1).sum():,}")
print(f"  Non-Norm (0)   : {(merged['label']==0).sum():,}")

print(f"\n  Source breakdown:")
for src, cnt in merged["source"].value_counts().items():
    print(f"    {src:35s}: {cnt:,}")

merged["word_count"] = merged["sentence"].str.split().str.len()
merged["char_count"] = merged["sentence"].str.len()

print(f"\n  Sentence word count stats:")
display(merged.groupby("label")["word_count"]
        .describe()[["mean","std","min","50%","max"]]
        .rename(index={0:"Non-Norm", 1:"Norm"})
        .round(1))

## 3. Label & Source Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Label & Source Distribution", fontsize=13, fontweight="bold")

# label bar
label_counts = merged["label"].value_counts().sort_index()
axes[0].bar(["Non-Norm (0)", "Norm (1)"], label_counts.values,
            color=[NONNORM_COLOR, NORM_COLOR], edgecolor="white", width=0.5)
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 100, f"{v:,}\n({v/len(merged)*100:.1f}%)", ha="center", fontsize=10)
axes[0].set_title("Label Distribution"); axes[0].set_ylabel("Count")
axes[0].set_ylim(0, max(label_counts.values) * 1.2)

# source bar
src_counts = merged["source"].value_counts()
colors = [NORM_COLOR if "culture" in s else NONNORM_COLOR for s in src_counts.index]
axes[1].barh(src_counts.index, src_counts.values, color=colors, edgecolor="white")
for i, v in enumerate(src_counts.values):
    axes[1].text(v + 30, i, f"{v:,}", va="center", fontsize=9)
axes[1].set_title("Source Distribution"); axes[1].set_xlabel("Count")
axes[1].invert_yaxis()

# pie
axes[2].pie(label_counts.values, labels=["Non-Norm", "Norm"],
            colors=[NONNORM_COLOR, NORM_COLOR],
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=2))
axes[2].set_title("Class Balance")

plt.tight_layout(); plt.show()

## 4. Sentence Length Analysis

In [ ]:
merged["label_name"] = merged["label"].map({0: "Non-Norm", 1: "Norm"})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Sentence Length Analysis", fontsize=13, fontweight="bold")

# word count histogram
for label, color, name in [(1, NORM_COLOR, "Norm"), (0, NONNORM_COLOR, "Non-Norm")]:
    axes[0,0].hist(merged[merged["label"]==label]["word_count"],
                   bins=40, alpha=0.7, color=color, label=name, edgecolor="white")
axes[0,0].set_title("Word Count Distribution"); axes[0,0].set_xlabel("Words")
axes[0,0].set_ylabel("Frequency"); axes[0,0].legend()

# char count histogram
for label, color, name in [(1, NORM_COLOR, "Norm"), (0, NONNORM_COLOR, "Non-Norm")]:
    axes[0,1].hist(merged[merged["label"]==label]["char_count"],
                   bins=40, alpha=0.7, color=color, label=name, edgecolor="white")
axes[0,1].set_title("Character Count Distribution"); axes[0,1].set_xlabel("Characters")
axes[0,1].set_ylabel("Frequency"); axes[0,1].legend()

# box plot
sns.boxplot(data=merged, x="label_name", y="word_count",
            palette={"Norm": NORM_COLOR, "Non-Norm": NONNORM_COLOR}, ax=axes[1,0])
axes[1,0].set_title("Word Count by Label"); axes[1,0].set_xlabel(""); axes[1,0].set_ylabel("Words")

# stats table
stats = merged.groupby("label_name")["word_count"].agg(["mean","std","min","median","max"]).round(1)
axes[1,1].axis("off")
tbl = axes[1,1].table(cellText=stats.values, rowLabels=stats.index,
                       colLabels=["Mean","Std","Min","Median","Max"],
                       loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.4, 2.4)
axes[1,1].set_title("Word Count Statistics", pad=20)

plt.tight_layout(); plt.show()

## 5. Cultural Group Analysis

In [ ]:
norms = merged[(merged["label"] == 1) & (merged["cultural_group"] != "none")].copy()
norms["cultural_group"] = norms["cultural_group"].str.strip().str.title()
top30 = norms["cultural_group"].value_counts().head(30)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Cultural Group Analysis (Norm Sentences Only)", fontsize=13, fontweight="bold")

# top 30 bar
axes[0].barh(top30.index[::-1], top30.values[::-1], color=NORM_COLOR, edgecolor="white", alpha=0.85)
for i, v in enumerate(top30.values[::-1]):
    axes[0].text(v + 3, i, str(v), va="center", fontsize=8)
axes[0].set_title("Top 30 Cultural Groups"); axes[0].set_xlabel("Count")

# top 15 — Reddit vs TikTok breakdown
top15 = top30.index[:15]
sub = norms[norms["cultural_group"].isin(top15)]
pivot = sub.groupby(["cultural_group", "source"]).size().unstack(fill_value=0).reindex(top15)
pivot.plot(kind="barh", ax=axes[1], color=[NORM_COLOR, "#90CAF9"], edgecolor="white")
axes[1].set_title("Top 15 Groups: Reddit vs TikTok")
axes[1].set_xlabel("Count"); axes[1].invert_yaxis()
axes[1].legend(title="Source")

plt.tight_layout(); plt.show()

print(f"\nTotal unique cultural groups: {norms['cultural_group'].nunique():,}")

## 6. Word Frequency Analysis

In [ ]:
STOPWORDS = {
    "the","a","an","is","are","was","were","be","been","being",
    "have","has","had","do","does","did","will","would","could",
    "should","may","might","to","of","in","on","at","for","with",
    "and","or","but","so","if","as","by","from","into","that",
    "this","it","its","they","their","there","which","who","when",
    "not","also","often","usually","typically","generally","can",
    "more","most","many","some","such","other","these","those",
    "each","than","then","about","up","out","no","any","all","both",
    "i","we","you","he","she","his","her","our","your","my",
}

def tokenize(text):
    return [w for w in re.findall(r"[a-z]+", str(text).lower())
            if w not in STOPWORDS and len(w) > 2]

norm_words    = Counter()
nonnorm_words = Counter()
for _, row in merged.iterrows():
    words = tokenize(row["sentence"])
    (norm_words if row["label"] == 1 else nonnorm_words).update(words)

top_n = 25
norm_top    = pd.DataFrame(norm_words.most_common(top_n),    columns=["word","count"])
nonnorm_top = pd.DataFrame(nonnorm_words.most_common(top_n), columns=["word","count"])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Top 25 Words (Stopwords Removed)", fontsize=13, fontweight="bold")

axes[0].barh(norm_top["word"][::-1],    norm_top["count"][::-1],    color=NORM_COLOR,    edgecolor="white")
axes[0].set_title("Norm Sentences"); axes[0].set_xlabel("Frequency")

axes[1].barh(nonnorm_top["word"][::-1], nonnorm_top["count"][::-1], color=NONNORM_COLOR, edgecolor="white")
axes[1].set_title("Non-Norm Sentences"); axes[1].set_xlabel("Frequency")

plt.tight_layout(); plt.show()

## 7. Word Clouds

In [ ]:
if HAS_WORDCLOUD:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Word Clouds", fontsize=13, fontweight="bold")

    for ax, word_freq, title, color in [
        (axes[0], dict(norm_words.most_common(200)),    "Norm Sentences",     NORM_COLOR),
        (axes[1], dict(nonnorm_words.most_common(200)), "Non-Norm Sentences", NONNORM_COLOR),
    ]:
        wc = WordCloud(width=700, height=400, background_color="white",
                       color_func=lambda *a, **kw: color,
                       max_words=150, prefer_horizontal=0.8
                       ).generate_from_frequencies(word_freq)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off"); ax.set_title(title, fontsize=12)

    plt.tight_layout(); plt.show()
else:
    print("Install wordcloud to see this plot: pip install wordcloud")

## 8. Train / Val / Test Split Overview

In [ ]:
splits = {"Train": train, "Val": val, "Test": test}
total  = sum(len(d) for d in splits.values())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Train / Val / Test Split Overview", fontsize=13, fontweight="bold")

# sizes
sizes = [len(d) for d in splits.values()]
bars  = axes[0].bar(splits.keys(), sizes,
                    color=["#42A5F5","#66BB6A","#FFA726"], edgecolor="white", width=0.5)
for bar, size in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f"{size:,}\n({size/total*100:.0f}%)", ha="center", fontsize=10)
axes[0].set_title("Split Sizes"); axes[0].set_ylabel("Rows")
axes[0].set_ylim(0, max(sizes) * 1.2)

# label balance per split
split_data = [{"Split": n, "Norm": df["label"].sum(), "Non-Norm": (df["label"]==0).sum()}
              for n, df in splits.items()]
pd.DataFrame(split_data).set_index("Split").plot(
    kind="bar", ax=axes[1], color=[NORM_COLOR, NONNORM_COLOR], edgecolor="white", rot=0)
axes[1].set_title("Label Balance per Split"); axes[1].set_ylabel("Count")
axes[1].legend(title="Label")

# source pie in train
src = train["source"].value_counts()
axes[2].pie(src.values, labels=src.index, autopct="%1.1f%%", startangle=90,
            colors=["#42A5F5","#66BB6A","#FFA726","#AB47BC"],
            wedgeprops=dict(edgecolor="white", linewidth=2))
axes[2].set_title("Source Mix in Train")

plt.tight_layout(); plt.show()

## 9. Sample Sentences

In [ ]:
print("=" * 70)
print("  SAMPLE NORM SENTENCES  (label = 1)")
print("=" * 70)
sample_norms = merged[merged["label"] == 1][["sentence","cultural_group","source"]].sample(8, random_state=42)
for _, row in sample_norms.iterrows():
    print(f"  [{row['cultural_group']}] {row['sentence']}")
    print(f"    source: {row['source']}\n")

print("=" * 70)
print("  SAMPLE NON-NORM SENTENCES  (label = 0)")
print("=" * 70)
sample_nonnorms = merged[merged["label"] == 0][["sentence","source"]].sample(8, random_state=42)
for _, row in sample_nonnorms.iterrows():
    print(f"  {row['sentence']}")
    print(f"    source: {row['source']}\n")